## Data Acquisition: News API

In [17]:
import os
import re
import sys
import html

import pandas as pd

from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from newsapi import NewsApiClient

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [18]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [19]:
api_key = os.getenv('NEWS_API_KEY')

In [20]:
QUERY_CONFIGS = [
    {"query_domain": "finance", "query": "(earnings OR revenue OR inflation OR interest rates) AND (expected OR forecast OR outlook OR projected)"},
    {"query_domain": "health", "query": "(disease OR virus OR vaccine OR public health) AND (expected OR forecast OR projected OR prognosis)"},
    {"query_domain": "weather", "query": "(weather OR storm OR hurricane OR rainfall OR temperature) AND (forecast OR expected OR outlook)"},
    {"query_domain": "policy", "query": "(election OR vote OR polling OR legislation) AND (expected OR likely OR projected OR forecast)"},
    {"query_domain": "sports_nfl", "query": "NFL AND (playoffs OR draft OR season) AND (expected OR likely OR prediction OR forecast)"},
    {"query_domain": "sports_nba", "query": "NBA AND (playoffs OR finals OR season) AND (expected OR likely OR prediction OR forecast)"},
    {"query_domain": "sports_college", "query": "(NCAA OR college football OR March Madness) AND (expected OR likely OR prediction OR forecast)"},
    {"query_domain": "misc-tesla", "query": "Tesla OR SAE Level 4 autonomy"},
]

list_num = 2

In [21]:
# query_string = "Tesla OR SAE Level 4 autonomy"
start_date = "2026-03-21"
end_date = "2026-03-28"
newsapi = NewsApiClient(api_key=api_key)
all_articles = newsapi.get_everything(
    q=QUERY_CONFIGS[list_num]['query'],
    language="en",
    from_param=start_date,
    to=end_date,
    sort_by="relevancy",
    page_size=7
)

In [22]:
all_articles.keys()

dict_keys(['status', 'totalResults', 'articles'])

In [23]:
len(all_articles['articles']), all_articles['articles']

(6,
 [{'source': {'id': None, 'name': 'BBC News'},
   'author': 'Simon King',
   'title': 'Northern Lights forecast to reappear across UK on Saturday',
   'description': "Forecasters say there's another chance to see the aurora tonight with solar activity remaining high.",
   'url': 'https://www.bbc.com/weather/articles/ckgwmr9ve10o',
   'urlToImage': 'https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/fcae/live/585cab30-24fd-11f1-ba46-637c713acadc.jpg',
   'publishedAt': '2026-03-21T08:57:17Z',
   'content': 'If you missed the display on Friday night, forecasters at the Met Office Space Weather Prediction Centre say it could be seen again on Saturday night.\r\nThey say "geomagnetic activity is expected to r… [+726 chars]'},
  {'source': {'id': None, 'name': 'BBC News'},
   'author': 'Stav Danaos',
   'title': 'Snow forecast for some as UK braces for cold snap',
   'description': "It's all change for the weather this week as temperatures plummet and snow is back in the forecas

In [24]:
df = pd.DataFrame(all_articles['articles'])
df

,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'BBC News'}",Simon King,Northern Lights forecast to reappear across UK on Saturday,Forecasters say there's another chance to see the aurora tonight with solar activity remaining high.,https://www.bbc.com/weather/articles/ckgwmr9ve10o,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/fcae/live/585cab30-24fd-11f1-ba46-637c713acadc.jpg,2026-03-21T08:57:17Z,"If you missed the display on Friday night, forecasters at the Met Office Space Weather Prediction Centre say it could be seen again on Saturday night.\r\nThey say ""geomagnetic activity is expected to r… [+726 chars]"
1,"{'id': None, 'name': 'BBC News'}",Stav Danaos,Snow forecast for some as UK braces for cold snap,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,https://www.bbc.com/weather/articles/cdxdvk95kkwo,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/5439/live/b308cdf0-2694-11f1-a79a-77e93010d956.jpg,2026-03-24T07:48:22Z,"From Tuesday, low pressure from the Atlantic will bring stronger winds and a band of rain sweeping across the UK.\r\nIt will be heavy at times in north-west England and North Wales. As the rain moves s… [+1140 chars]"
2,"{'id': None, 'name': 'BBC News'}",Ben Rich,Fog and sunshine in the weekend forecast as high pressure clings on,After a week that brought the warmest weather of the year so far temperatures are set to drop a little but many places will remain dry as Ben Rich explains.,https://www.bbc.com/weather/articles/c74vjdzk2xko,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/1e93/live/4e622690-2468-11f1-a2c1-39d16ad974aa.jpg,2026-03-21T03:00:39Z,"As we move beyond the weekend, the weather looks set to give us a reminder that we are still in early spring - with high pressure expected to decline and chilly conditions due to make a return.\r\nAfte… [+510 chars]"
3,"{'id': None, 'name': 'mlive.com'}",Tanda Gmiter,Egg-sized hail possible with incoming storms in part of Michigan,Severe weather is expected largely across southern Michigan with Thursday night's storms.,https://www.mlive.com/weather/2026/03/egg-sized-hail-possible-with-incoming-storms-in-part-of-michigan.html,https://s.yimg.com/ny/api/res/1.2/dxuqafAw9AB0Z31MIRGtVA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD05MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/mlive_com_articles_368/09eccbd48479ba2573cf6d073b3171f4,2026-03-26T19:58:17Z,Hail as big as hens eggs is a possibility with Thursday nights incoming storm system that is forecast to develop over the southern tier of Michigans Lower Peninsula.\r\nWhile there is also a risk of th… [+1157 chars]
4,"{'id': None, 'name': 'Idaho Statesman'}",Hali Smith,"Rain, mountain snow to break Boise’s warm streak. When will temperatures drop?","Idaho is about to undergo a “significant pattern shift,” the National Weather Service said.",https://www.idahostatesman.com/news/weather-news/article315212931.html,https://s.yimg.com/os/en/idaho_statesman_mcclatchy_articles_842/2b75e51e132557ba2918148bc6d5bc47,2026-03-27T21:23:30Z,"Heres some good news if youre headed to Boises Treefort Music Fest.\r\nThe City of Trees should be sunny and warm all weekend long, according to the National Weather Service.\r\nRecord or near-record hig… [+3048 chars]"
5,"{'id': None, 'name': 'Gizmodo.com'}","Jake Bittle, Emily Jones, Juanpablo Ramirez-Franco, Vivian La, Anila Yoganathan, Katie Myers, & Clayton Aldern, Grist",Your Home Could Soon Be Uninsurable If You’re in One of These States,Home insurance is buckling under climate risk and construction trends. Find out how your state fares.,https://gizmodo.com/your-home-could-soon-be-uninsurable-if-youre-in-one-of-these-states-2000736848,https://gizmodo.com/app/uploads/2026/03/oklahomahouse-1200x675.jpg,2026-03-23T14:55:14Z,"In recent years, as the United States has suffered a series of damaging climate disasters, experts have warned that the nation is headed toward a 

In [25]:
df = df.rename(columns={
    "source": "Source Meta Data",
    "author": "Source",
    "publishedAt": "Date",
    "url": "URL",
    "urlToImage": "Image URL"
})

df["Query Domain"] = QUERY_CONFIGS[list_num]["query_domain"]

df

,Source Meta Data,Source,title,description,URL,Image URL,Date,content,Query Domain
0,"{'id': None, 'name': 'BBC News'}",Simon King,Northern Lights forecast to reappear across UK on Saturday,Forecasters say there's another chance to see the aurora tonight with solar activity remaining high.,https://www.bbc.com/weather/articles/ckgwmr9ve10o,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/fcae/live/585cab30-24fd-11f1-ba46-637c713acadc.jpg,2026-03-21T08:57:17Z,"If you missed the display on Friday night, forecasters at the Met Office Space Weather Prediction Centre say it could be seen again on Saturday night.\r\nThey say ""geomagnetic activity is expected to r… [+726 chars]",weather
1,"{'id': None, 'name': 'BBC News'}",Stav Danaos,Snow forecast for some as UK braces for cold snap,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,https://www.bbc.com/weather/articles/cdxdvk95kkwo,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/5439/live/b308cdf0-2694-11f1-a79a-77e93010d956.jpg,2026-03-24T07:48:22Z,"From Tuesday, low pressure from the Atlantic will bring stronger winds and a band of rain sweeping across the UK.\r\nIt will be heavy at times in north-west England and North Wales. As the rain moves s… [+1140 chars]",weather
2,"{'id': None, 'name': 'BBC News'}",Ben Rich,Fog and sunshine in the weekend forecast as high pressure clings on,After a week that brought the warmest weather of the year so far temperatures are set to drop a little but many places will remain dry as Ben Rich explains.,https://www.bbc.com/weather/articles/c74vjdzk2xko,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/1e93/live/4e622690-2468-11f1-a2c1-39d16ad974aa.jpg,2026-03-21T03:00:39Z,"As we move beyond the weekend, the weather looks set to give us a reminder that we are still in early spring - with high pressure expected to decline and chilly conditions due to make a return.\r\nAfte… [+510 chars]",weather
3,"{'id': None, 'name': 'mlive.com'}",Tanda Gmiter,Egg-sized hail possible with incoming storms in part of Michigan,Severe weather is expected largely across southern Michigan with Thursday night's storms.,https://www.mlive.com/weather/2026/03/egg-sized-hail-possible-with-incoming-storms-in-part-of-michigan.html,https://s.yimg.com/ny/api/res/1.2/dxuqafAw9AB0Z31MIRGtVA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD05MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/mlive_com_articles_368/09eccbd48479ba2573cf6d073b3171f4,2026-03-26T19:58:17Z,Hail as big as hens eggs is a possibility with Thursday nights incoming storm system that is forecast to develop over the southern tier of Michigans Lower Peninsula.\r\nWhile there is also a risk of th… [+1157 chars],weather
4,"{'id': None, 'name': 'Idaho Statesman'}",Hali Smith,"Rain, mountain snow to break Boise’s warm streak. When will temperatures drop?","Idaho is about to undergo a “significant pattern shift,” the National Weather Service said.",https://www.idahostatesman.com/news/weather-news/article315212931.html,https://s.yimg.com/os/en/idaho_statesman_mcclatchy_articles_842/2b75e51e132557ba2918148bc6d5bc47,2026-03-27T21:23:30Z,"Heres some good news if youre headed to Boises Treefort Music Fest.\r\nThe City of Trees should be sunny and warm all weekend long, according to the National Weather Service.\r\nRecord or near-record hig… [+3048 chars]",weather
5,"{'id': None, 'name': 'Gizmodo.com'}","Jake Bittle, Emily Jones, Juanpablo Ramirez-Franco, Vivian La, Anila Yoganathan, Katie Myers, & Clayton Aldern, Grist",Your Home Could Soon Be Uninsurable If You’re in One of These States,Home insurance is buckling under climate risk and construction trends. Find out how your state fares.,https://gizmodo.com/your-home-could-soon-be-uninsurable-if-youre-in-one-of-these-states-2000736848,https://gizmodo.com/app/uploads/2026/03/oklahomahouse-1200x675.jpg,2026-03-23T14:55:14Z,"In recent years, as the United States has suffered a series of damaging climate disasters, 

In [26]:
TRUNC_PATTERN = re.compile(r"(\u2026|\.\.\.)\s*\[\+\d+\s+chars\]")

def clean_base_sentence(text: str) -> str:
    if not isinstance(text, str):
        return text

    # ✅ REMOVE NewsAPI truncation marker (ONLY in cleaned version)
    text = TRUNC_PATTERN.sub("", text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove UL/LI tags
    text = re.sub(r"</?ul>", "", text)
    text = re.sub(r"</?li>", " ", text)

    # Normalize whitespace
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [27]:
def build_prediction_dataframe(df):
    df = df.reset_index(drop=True).copy()
    df["article_id"] = df.index

    prediction_keywords = [
        "expected", "forecast", "forecasted", "forecasting",
        "forecast estimate", "forecasted outcome", "projected",
        "projection", "predict", "predicted", "estimate",
        "expectation", "anticipation", "prognosis",
        "speculation", "foretelling", "prophecy", "guess"
    ]

    pattern = r"\b(" + "|".join(map(re.escape, prediction_keywords)) + r")\b"

    df["prediction_visible"] = (
        (
            df["title"].fillna("") +
            df["description"].fillna("") +
            df["content"].fillna("")
        )
        .str.lower()
        .str.contains(pattern, regex=True)
    ).astype(int)

    parts = []
    for field, order in [("title", 0), ("description", 1), ("content", 2)]:
        tmp = df.copy()
        tmp["Base Sentence (raw)"] = tmp[field]   # ✅ RAW PRESERVED
        tmp["Source_Field"] = field
        tmp["field_order"] = order
        parts.append(tmp)

    long_df = pd.concat(parts, ignore_index=True)

    # ✅ CLEANED COLUMN (used downstream)
    long_df["Base Sentence"] = long_df["Base Sentence (raw)"].apply(clean_base_sentence)

    # ✅ Sentence splitting uses CLEAN text
    sfe = SpacyFeatureExtraction(long_df, "Base Sentence")
    long_df = sfe.split_into_sentences()

    # ✅ Label per sentence
    long_df["Sentence Label"] = (
        long_df["Base Sentence"]
        .str.lower()
        .str.contains(pattern, regex=True)
        .astype(int)
    )

    long_df["Human Annotation"] = ""
    long_df["Human Reasoning"] = ""

    priority_cols = [
        "Base Sentence",
        "Base Sentence (raw)",
        "Sentence Label",
        "Human Annotation",
        "Human Reasoning",
        "Source",
        "Date"
    ]

    priority_cols = [c for c in priority_cols if c in long_df.columns]
    remaining_cols = [c for c in long_df.columns if c not in priority_cols]

    return long_df[priority_cols + remaining_cols]

In [28]:
final_df = build_prediction_dataframe(df)
final_df.head(3)

/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_58652/543514960.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, regex=True)
/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_58652/543514960.py:46: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, regex=True)


,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
0,Northern Lights forecast to reappear across UK on Saturday,Northern Lights forecast to reappear across UK on Saturday,1,,,Simon King,2026-03-21T08:57:17Z,"{'id': None, 'name': 'BBC News'}",Northern Lights forecast to reappear across UK on Saturday,Forecasters say there's another chance to see the aurora tonight with solar activity remaining high.,https://www.bbc.com/weather/articles/ckgwmr9ve10o,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/fcae/live/585cab30-24fd-11f1-ba46-637c713acadc.jpg,"If you missed the display on Friday night, forecasters at the Met Office Space Weather Prediction Centre say it could be seen again on Saturday night.\r\nThey say ""geomagnetic activity is expected to r… [+726 chars]",weather,0,1,title,0
1,Snow forecast for some as UK braces for cold snap,Snow forecast for some as UK braces for cold snap,1,,,Stav Danaos,2026-03-24T07:48:22Z,"{'id': None, 'name': 'BBC News'}",Snow forecast for some as UK braces for cold snap,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,https://www.bbc.com/weather/articles/cdxdvk95kkwo,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/5439/live/b308cdf0-2694-11f1-a79a-77e93010d956.jpg,"From Tuesday, low pressure from the Atlantic will bring stronger winds and a band of rain sweeping across the UK.\r\nIt will be heavy at times in north-west England and North Wales. As the rain moves s… [+1140 chars]",weather,1,1,title,0
2,Fog and sunshine in the weekend forecast as high pressure clings on,Fog and sunshine in the weekend forecast as high pressure clings on,1,,,Ben Rich,2026-03-21T03:00:39Z,"{'id': None, 'name': 'BBC News'}",Fog and sunshine in the weekend forecast as high pressure clings on,After a week that brought the warmest weather of the year so far temperatures are set to drop a little but many places will remain dry as Ben Rich explains.,https://www.bbc.com/weather/articles/c74vjdzk2xko,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/1e93/live/4e622690-2468-11f1-a2c1-39d16ad974aa.jpg,"As we move beyond the weekend, the weather looks set to give us a reminder that we are still in early spring - with high pressure expected to decline and chilly conditions due to make a return.\r\nAfte… [+510 chars]",weather,2,1,title,0


In [29]:
predictions_df = final_df[final_df["Sentence Label"] == 1]
predictions_df.head(3)

,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
0,Northern Lights forecast to reappear across UK on Saturday,Northern Lights forecast to reappear across UK on Saturday,1,,,Simon King,2026-03-21T08:57:17Z,"{'id': None, 'name': 'BBC News'}",Northern Lights forecast to reappear across UK on Saturday,Forecasters say there's another chance to see the aurora tonight with solar activity remaining high.,https://www.bbc.com/weather/articles/ckgwmr9ve10o,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/fcae/live/585cab30-24fd-11f1-ba46-637c713acadc.jpg,"If you missed the display on Friday night, forecasters at the Met Office Space Weather Prediction Centre say it could be seen again on Saturday night.\r\nThey say ""geomagnetic activity is expected to r… [+726 chars]",weather,0,1,title,0
1,Snow forecast for some as UK braces for cold snap,Snow forecast for some as UK braces for cold snap,1,,,Stav Danaos,2026-03-24T07:48:22Z,"{'id': None, 'name': 'BBC News'}",Snow forecast for some as UK braces for cold snap,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,https://www.bbc.com/weather/articles/cdxdvk95kkwo,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/5439/live/b308cdf0-2694-11f1-a79a-77e93010d956.jpg,"From Tuesday, low pressure from the Atlantic will bring stronger winds and a band of rain sweeping across the UK.\r\nIt will be heavy at times in north-west England and North Wales. As the rain moves s… [+1140 chars]",weather,1,1,title,0
2,Fog and sunshine in the weekend forecast as high pressure clings on,Fog and sunshine in the weekend forecast as high pressure clings on,1,,,Ben Rich,2026-03-21T03:00:39Z,"{'id': None, 'name': 'BBC News'}",Fog and sunshine in the weekend forecast as high pressure clings on,After a week that brought the warmest weather of the year so far temperatures are set to drop a little but many places will remain dry as Ben Rich explains.,https://www.bbc.com/weather/articles/c74vjdzk2xko,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/1e93/live/4e622690-2468-11f1-a2c1-39d16ad974aa.jpg,"As we move beyond the weekend, the weather looks set to give us a reminder that we are still in early spring - with high pressure expected to decline and chilly conditions due to make a return.\r\nAfte… [+510 chars]",weather,2,1,title,0


In [30]:
predictions_df["Base Sentence"] = predictions_df["Base Sentence"].apply(clean_base_sentence)
predictions_df.head(3)

,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
0,Northern Lights forecast to reappear across UK on Saturday,Northern Lights forecast to reappear across UK on Saturday,1,,,Simon King,2026-03-21T08:57:17Z,"{'id': None, 'name': 'BBC News'}",Northern Lights forecast to reappear across UK on Saturday,Forecasters say there's another chance to see the aurora tonight with solar activity remaining high.,https://www.bbc.com/weather/articles/ckgwmr9ve10o,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/fcae/live/585cab30-24fd-11f1-ba46-637c713acadc.jpg,"If you missed the display on Friday night, forecasters at the Met Office Space Weather Prediction Centre say it could be seen again on Saturday night.\r\nThey say ""geomagnetic activity is expected to r… [+726 chars]",weather,0,1,title,0
1,Snow forecast for some as UK braces for cold snap,Snow forecast for some as UK braces for cold snap,1,,,Stav Danaos,2026-03-24T07:48:22Z,"{'id': None, 'name': 'BBC News'}",Snow forecast for some as UK braces for cold snap,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,https://www.bbc.com/weather/articles/cdxdvk95kkwo,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/5439/live/b308cdf0-2694-11f1-a79a-77e93010d956.jpg,"From Tuesday, low pressure from the Atlantic will bring stronger winds and a band of rain sweeping across the UK.\r\nIt will be heavy at times in north-west England and North Wales. As the rain moves s… [+1140 chars]",weather,1,1,title,0
2,Fog and sunshine in the weekend forecast as high pressure clings on,Fog and sunshine in the weekend forecast as high pressure clings on,1,,,Ben Rich,2026-03-21T03:00:39Z,"{'id': None, 'name': 'BBC News'}",Fog and sunshine in the weekend forecast as high pressure clings on,After a week that brought the warmest weather of the year so far temperatures are set to drop a little but many places will remain dry as Ben Rich explains.,https://www.bbc.com/weather/articles/c74vjdzk2xko,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/1e93/live/4e622690-2468-11f1-a2c1-39d16ad974aa.jpg,"As we move beyond the weekend, the weather looks set to give us a reminder that we are still in early spring - with high pressure expected to decline and chilly conditions due to make a return.\r\nAfte… [+510 chars]",weather,2,1,title,0


In [31]:
predictions_df

,Base Sentence,Base Sentence (raw),Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
0,Northern Lights forecast to reappear across UK on Saturday,Northern Lights forecast to reappear across UK on Saturday,1,,,Simon King,2026-03-21T08:57:17Z,"{'id': None, 'name': 'BBC News'}",Northern Lights forecast to reappear across UK on Saturday,Forecasters say there's another chance to see the aurora tonight with solar activity remaining high.,https://www.bbc.com/weather/articles/ckgwmr9ve10o,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/fcae/live/585cab30-24fd-11f1-ba46-637c713acadc.jpg,"If you missed the display on Friday night, forecasters at the Met Office Space Weather Prediction Centre say it could be seen again on Saturday night.\r\nThey say ""geomagnetic activity is expected to r… [+726 chars]",weather,0,1,title,0
1,Snow forecast for some as UK braces for cold snap,Snow forecast for some as UK braces for cold snap,1,,,Stav Danaos,2026-03-24T07:48:22Z,"{'id': None, 'name': 'BBC News'}",Snow forecast for some as UK braces for cold snap,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,https://www.bbc.com/weather/articles/cdxdvk95kkwo,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/5439/live/b308cdf0-2694-11f1-a79a-77e93010d956.jpg,"From Tuesday, low pressure from the Atlantic will bring stronger winds and a band of rain sweeping across the UK.\r\nIt will be heavy at times in north-west England and North Wales. As the rain moves s… [+1140 chars]",weather,1,1,title,0
2,Fog and sunshine in the weekend forecast as high pressure clings on,Fog and sunshine in the weekend forecast as high pressure clings on,1,,,Ben Rich,2026-03-21T03:00:39Z,"{'id': None, 'name': 'BBC News'}",Fog and sunshine in the weekend forecast as high pressure clings on,After a week that brought the warmest weather of the year so far temperatures are set to drop a little but many places will remain dry as Ben Rich explains.,https://www.bbc.com/weather/articles/c74vjdzk2xko,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/1e93/live/4e622690-2468-11f1-a2c1-39d16ad974aa.jpg,"As we move beyond the weekend, the weather looks set to give us a reminder that we are still in early spring - with high pressure expected to decline and chilly conditions due to make a return.\r\nAfte… [+510 chars]",weather,2,1,title,0
8,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,1,,,Stav Danaos,2026-03-24T07:48:22Z,"{'id': None, 'name': 'BBC News'}",Snow forecast for some as UK braces for cold snap,It's all change for the weather this week as temperatures plummet and snow is back in the forecast.,https://www.bbc.com/weather/articles/cdxdvk95kkwo,https://ichef.bbci.co.uk/ace/branded_weather/1200/cpsprodpb/5439/live/b308cdf0-2694-11f1-a79a-77e93010d956.jpg,"From Tuesday, low pressure from the Atlantic will bring stronger winds and a band of rain sweeping across the UK.\r\nIt will be heavy at times in north-west England and North Wales. As the rain moves s… [+1140 chars]",weather,1,1,description,1
10,Severe weather is expected largely across southern Michigan with Thursday night's storms.,Severe weather is expected largely across southern Michigan with Thursday night's storms.,1,,,Tanda Gmiter,2026-03-26T19:58:17Z,"{'id': None, 'name': 'mlive.com'}",Egg-sized hail possible with incoming storms in part of Michigan,Severe weather is expected largely across southern Michigan with Thursday night's storms.,https://www.mlive.com/weather/2026/03/egg-sized-hail-possible-with-incoming-storms-in-part-of-michigan.html,https://s.yimg.com/ny/api/res/1.2/dxuqafAw9AB0Z31MIRGtVA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD05MDA7Y2Y9d2VicA--/https://medi

In [32]:
len(predictions_df)

8

In [ ]:
def build_query_package(
    *,
    query_string: str,
    query_domain: str, 
    start_date: str,
    end_date: str,
    df,
    final_df,
    predictions_df,
    language: str = "en",
    sort_by: str = "relevancy",
    page_size: int = None
):
    """
    Builds a query-aligned package and a deterministic filename prefix.
    """

    # -------------------------
    # Prefix from query
    # -------------------------
    base_prefix = (
        query_string.lower()
        .replace(" or ", "_")
        .replace(" and ", "_")
        .replace(" ", "_")
    )
    base_prefix = re.sub(r"[^a-z0-9_]", "", base_prefix)
    base_prefix = re.sub(r"_+", "_", base_prefix).strip("_")

    prefix = f"{base_prefix}_{start_date}_to_{end_date}"

    # -------------------------
    # Query metadata
    # -------------------------
    query_meta = {
        "query": query_string,
        "query_domain": query_domain,
        "language": language,
        "from_date": start_date,
        "to_date": end_date,
        "sort_by": sort_by,
        "page_size": page_size,
        "timestamp_utc": datetime.utcnow().isoformat()
    }

    # -------------------------
    # Full package
    # -------------------------
    package = {
        "query_meta": query_meta,
        "counts": {
            "num_raw_articles": len(df),
            "num_sentences": len(final_df),
            "num_potential_predictions": len(predictions_df)
        },
        "raw_articles": df.to_dict(orient="records"),
        "processed_sentences": final_df.to_dict(orient="records"),
        "potential_predictions": predictions_df.to_dict(orient="records")
    }

    return prefix, package

In [ ]:
prefix, query_package = build_query_package(
    query_string=QUERY_CONFIGS[list_num]["query"],
    query_domain=QUERY_CONFIGS[list_num]["query_domain"],
    start_date=start_date,
    end_date=end_date,
    df=df,
    final_df=final_df,
    predictions_df=predictions_df,
    page_size=7
)

In [ ]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
save_path = os.path.join(base_data_path, 'news_api', 'annotators')

DataProcessing.save_to_file(
    data=predictions_df,
    path=save_path,
    prefix=f"{prefix}_predictions",
    save_file_type="csv"
)

In [ ]:
DataProcessing.save_to_file(
    data=query_package,
    path=save_path,
    prefix=f"{prefix}_full",
    save_file_type="json"
)